In [3]:
import re
import os
import csv
os.getcwd()

'/Users/Angel/Desktop/prism-games/prism-examples/csgs/learning/analysis'

In [12]:
import re
import os
import csv

# ===== INPUT / OUTPUT =====
output_dir = os.getcwd() + "/../results/"
base_name = "traffic_merge1"
log_file = output_dir + base_name  # change if needed
output_file = output_dir + base_name + ".csv"

os.makedirs(output_dir, exist_ok=True)

In [13]:
# ===== READ LOG =====
with open(log_file, "r") as f:
    text = f.read()

# ===== HELPERS =====
def extract(pattern, group=1, default=""):
    m = re.search(pattern, text, re.DOTALL)
    return m.group(group).strip() if m else default

def clean_strategy(s):
    return s.replace("\n", " ").strip()

# ===== NEW FRONT FIELDS =====
p_reach = extract(r"Lower bound on reachability probability:\s*([0-9\.Ee\-]+)")
p_T = extract(r"Stopping probability:\s*([0-9\.Ee\-]+)")
eff_horizon = extract(r"Effective horizon:\s*([0-9]+)")
eps = extract(r"Epsilon:\s*([0-9\.Ee\-]+)")
confidence = extract(r"Confidence:\s*([0-9\.Ee\-]+)")

# ===== GLOBAL =====
exec_time = extract(r"Execution time:\s*([0-9\.]+)")
episodes = extract(r"Episodes=([0-9]+)")
deltaT = extract(r"DeltaT=([0-9\.Ee\-]+)")
nMin = extract(r"nMin=([0-9]+)")

# ===== ROBUST =====
robust_found = extract(r"Robust SolveOutcome\{found\s*=\s*(true|false)").upper()
robust_value = extract(r"Robust SolveOutcome\{.*?value\s*=\s*([0-9\.Ee\-]+)")
robust_strategy = clean_strategy(extract(r"Robust SolveOutcome\{.*?strategy\s*=\s*(CSG:.*?)\}"))

robust_true_val = extract(r"Evaluating robust strategy.*?True value of learned strategy:\s*([0-9\.Ee\-]+)")
robust_gap = extract(r"Evaluating robust strategy.*?Value gap:\s*([0-9\.Ee\-]+)")
robust_dev = extract(r"Evaluating robust strategy.*?Max deviation gain of learned strategy:\s*([0-9\.Ee\-]+)")

# ===== POINT =====
point_found = extract(r"Point SolveOutcome\{found\s*=\s*(true|false)").upper()
point_value = extract(r"Point SolveOutcome\{.*?value\s*=\s*([0-9\.Ee\-]+)")
point_strategy = clean_strategy(extract(r"Point SolveOutcome\{.*?strategy\s*=\s*(CSG:.*?)\}"))

point_true_val = extract(r"Evaluating point strategy.*?True value of learned strategy:\s*([0-9\.Ee\-]+)")
point_gap = extract(r"Evaluating point strategy.*?Value gap:\s*([0-9\.Ee\-]+)")
point_dev = extract(r"Evaluating point strategy.*?Max deviation gain of learned strategy:\s*([0-9\.Ee\-]+)")

# ===== TRUE =====
true_found = extract(r"True SolveOutcome\{found\s*=\s*(true|false)").upper()
true_value = extract(r"True SolveOutcome\{.*?value\s*=\s*([0-9\.Ee\-]+)")
true_strategy = clean_strategy(extract(r"True SolveOutcome\{.*?strategy\s*=\s*(CSG:.*?)\}"))
true_sw = extract(r"Coalition results \(initial state\):\s*(\([0-9\.,]+\))")

# ===== ROW =====
row = [
    float(p_reach) if p_reach else "",
    float(p_T) if p_T else "",
    float(eps) if eps else "",
    float(confidence) if confidence else "",
    int(eff_horizon) if eff_horizon else "",

    float(exec_time) if exec_time else "",
    int(episodes) if episodes else "",
    float(deltaT) if deltaT else "",
    int(nMin) if nMin else "",

    robust_found,
    float(robust_value) if robust_value else "",
    robust_strategy,
    float(robust_true_val) if robust_true_val else "",
    float(robust_gap) if robust_gap else "",
    float(robust_dev) if robust_dev else "",

    point_found,
    float(point_value) if point_value else "",
    point_strategy,
    float(point_true_val) if point_true_val else "",
    float(point_gap) if point_gap else "",
    float(point_dev) if point_dev else "",

    true_found,
    true_sw,
    float(true_value) if true_value else "",
    true_strategy
]

# ===== HEADER =====
header = [
    "p_reach", "p_T", "eps", "confidence", "effective horizon",
    "Execution time (ms)", "episodes", "deltaT", "nMin",
    "Robust foundNE", "Robust value", "Robust strategy",
    "Robust true value", "Robust value gap", "Robust dev gain",
    "Point foundNE", "Point value", "Point strategy",
    "Point true value", "Point value gap", "Point dev gain",
    "True found", "True value", "True SW", "True strategy"
]

# ===== WRITE CSV =====
file_exists = os.path.exists(output_file)

with open(output_file, "a", newline="") as f:
    writer = csv.writer(f)

    if not file_exists or os.stat(output_file).st_size == 0:
        writer.writerow(header)

    writer.writerow(row)

print("Parsed and written to:", output_file)

Parsed and written to: /Users/Angel/Desktop/prism-games/prism-examples/csgs/learning/analysis/../results/traffic_merge1.csv
